# Evaluation — Model Performance Analysis

This notebook evaluates the trained GRU and Transformer models on the held-out test set.

**Metrics:** Accuracy (Hit@1), Precision@K, Recall@K

**Prerequisites:** Run the Modeling notebook first to train and save model checkpoints to `models/`.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn

sys.path.insert(0, '..')
from src.models import GRURecommender, TransformerRecommender

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

PROCESSED = Path('../processed')
OUT       = Path('outputs'); OUT.mkdir(exist_ok=True)

_ts = datetime.now().strftime('%Y%m%d_%H%M')

# ── Load data ────────────────────────────────────────────────────────────────
vocab     = pd.read_parquet(PROCESSED / 'track_vocab.parquet')
test_seqs = pd.read_parquet(PROCESSED / 'test_seqs.parquet')
print(f'Vocab: {len(vocab):,} tracks')
print(f'Test:  {len(test_seqs):,} sequences')

# ── Parameters (must match training) ─────────────────────────────────────────
VOCAB_LIMIT = 50_000
MAX_SEQ_LEN = 50
EMBED_DIM   = 128
HIDDEN_DIM  = 256
NUM_LAYERS  = 2
NUM_HEADS   = 4
DROPOUT     = 0.2

PAD_IDX    = VOCAB_LIMIT
UNK_IDX    = VOCAB_LIMIT + 1
NUM_TOKENS = VOCAB_LIMIT + 2

SEED_RATIO = 0.8
K_VALUES   = [1, 5, 10, 20]

---
## 1. Load Trained Models

Instantiate both architectures (from `src/models.py`) and load the best checkpoints saved during training.

In [2]:
gru_model = GRURecommender(
    NUM_TOKENS, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, PAD_IDX
).to(device)
gru_model.load_state_dict(torch.load('../models/gru_best.pt', map_location=device, weights_only=True))
gru_model.eval()
print(f'GRU loaded         ({sum(p.numel() for p in gru_model.parameters()):,} params)')

transformer_model = TransformerRecommender(
    NUM_TOKENS, EMBED_DIM, NUM_HEADS, NUM_LAYERS, DROPOUT, MAX_SEQ_LEN, PAD_IDX
).to(device)
transformer_model.load_state_dict(torch.load('../models/transformer_best.pt', map_location=device, weights_only=True))
transformer_model.eval()
print(f'Transformer loaded  ({sum(p.numel() for p in transformer_model.parameters()):,} params)')

GRU loaded         (19,941,970 params)
Transformer loaded  (13,253,458 params)


---
## 2. Evaluation Metrics

We evaluate with the metrics specified in the assignment:
- **Accuracy** — top-1 next-track prediction accuracy (is the most probable next track the correct one?)
- **Precision@K** — of the top K predicted tracks, how many appear in the held-out continuation?
- **Recall@K** — of all held-out tracks, how many appear in the top K predictions?

**Protocol:** Each test playlist is split into a seed (first 80% of tracks) and a holdout (remaining 20%). The model receives the seed and its top-K predictions from the last position are compared against the holdout tracks.

In [3]:
@torch.no_grad()
def evaluate_recsys(model, seqs_df, vocab_limit, max_len, dev,
                    seed_ratio=0.8, ks=[1, 5, 10, 20], max_eval=5000):
    """Recommendation evaluation: seed -> predict -> compare with holdout."""
    model.eval()
    unk = vocab_limit + 1
    metrics = {'Accuracy': []}
    for k in ks:
        metrics[f'Prec@{k}'] = []
        metrics[f'Recall@{k}'] = []

    seqs = seqs_df['track_idxs'].tolist()
    if len(seqs) > max_eval:
        seqs = [seqs[i] for i in np.random.default_rng(42).choice(len(seqs), max_eval, replace=False)]

    n_eval = 0
    for idxs in seqs:
        mapped = [i if i < vocab_limit else unk for i in idxs]
        sp = max(1, int(len(mapped) * seed_ratio))
        if sp >= len(mapped) - 1:
            continue
        seed = mapped[:sp]
        if len(seed) > max_len:
            seed = seed[-max_len:]
        holdout = set(mapped[sp:]) - {unk}
        if not holdout:
            continue

        inp   = torch.tensor([seed], dtype=torch.long, device=dev)
        pmask = torch.zeros(1, len(seed), dtype=torch.bool, device=dev)
        logits = model(inp, pad_mask=pmask)[0, -1, :vocab_limit]
        topk   = logits.topk(max(ks)).indices.tolist()

        metrics['Accuracy'].append(1 if topk[0] == mapped[sp] else 0)
        for k in ks:
            hits = len(set(topk[:k]) & holdout)
            metrics[f'Prec@{k}'].append(hits / k)
            metrics[f'Recall@{k}'].append(hits / len(holdout))
        n_eval += 1

    print(f'  Evaluated {n_eval:,} playlists')
    return {m: np.mean(v) for m, v in metrics.items()}

---
## 3. Results

In [5]:
print('GRU:')
gru_metrics = evaluate_recsys(gru_model, test_seqs, VOCAB_LIMIT, MAX_SEQ_LEN, device, ks=K_VALUES)

print('Transformer:')
tf_metrics = evaluate_recsys(transformer_model, test_seqs, VOCAB_LIMIT, MAX_SEQ_LEN, device, ks=K_VALUES)

# add both metrics to result
results_df = pd.DataFrame({
    'Metric': list(gru_metrics.keys()),
    'GRU': list(gru_metrics.values()),
    'Transformer': list(tf_metrics.values())
})

GRU:
  Evaluated 4,756 playlists
Transformer:
  Evaluated 4,756 playlists


### 3.1 Interpretation

The table above compares both architectures across all evaluation metrics:

- **Accuracy (Hit@1)** measures exact next-track prediction — the hardest metric since the model must rank the correct track first among 50,000 candidates
- **Precision@K** reflects the fraction of recommendations that are relevant — higher K gives more chances but dilutes precision
- **Recall@K** is the most practically useful metric — it measures what fraction of the user's actual future listening the model captures in its top-K list
- GRU captures local sequential patterns while Transformer can model longer-range dependencies; their relative performance reflects which signal dominates in this dataset

---
## 4. Summary & Best Model

In [ ]:
print('Final Model Comparison')
print('=' * 70)
summary = pd.DataFrame({'GRU': gru_metrics, 'Transformer': tf_metrics}).T
print(summary.to_string(float_format='%.4f'))

gru_r10 = gru_metrics.get('Recall@10', 0)
tf_r10  = tf_metrics.get('Recall@10', 0)
best_name = 'GRU' if gru_r10 >= tf_r10 else 'Transformer'
print(f'\nBest model by Recall@10: {best_name}')
print(f'Checkpoint: models/{best_name.lower()}_best.pt')

results_path = f'../models/results_summary_{_ts}.csv'
summary.to_csv(results_path)
print(f'Results saved to {results_path}')

---
## 5. Conclusions

1. **Sequential models learn playlist patterns** — both GRU and Transformer capture sequential structure in playlists, producing meaningful next-track recommendations from learned track embeddings
2. **Power-law popularity distribution** shapes performance — popular tracks are easier to predict, while the long tail remains challenging
3. **Vocabulary truncation** to 50K tracks is effective — covers the vast majority of sequence tokens while keeping the model tractable
4. **Embedding dimension and sequence length** influence accuracy with diminishing returns (shown in ablation experiments in the Modeling notebook)
5. **GRU vs. Transformer** — both architectures are competitive; GRU is simpler and faster while Transformer can capture longer-range patterns
6. **Best model checkpoint** is saved for potential deployment in Mini-Project 3